# Notebook 2 : premier test du solveur d'advection

**Objectif** : vérifier que le solveur volumes finis (VF) marche sur un cas dont on connaît la réponse, *avant* de l'utiliser sur le labyrinthe.

**Le cas test** : une tache de concentration (une gaussienne) posée au milieu d'une grille vide, avec une vitesse uniforme qui pousse tout vers la droite. La bonne réponse est simple : la tache doit se déplacer vers la droite à la vitesse `u`, sans changer de forme.

**Ce qu'on vérifie à la fin** :
1. la tache se déplace bien de la bonne distance,
2. la masse totale reste constante (propriété de la forme conservative),
3. un champ constant reste constant (détecte les erreurs d'indexation),
4. on mesure l'erreur par rapport à la solution exacte, et on voit l'étalement dû à l'upwind.

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 2. Les fonctions du solveur

Les conventions sont les mêmes que dans ton script des vitesses :
- `c[i, j]` : concentration moyenne de la cellule (ligne `i`, colonne `j`), taille `(Ny, Nx)`.
- `U[i, j]` : vitesse sur la face **gauche** de la cellule `(i, j)`, taille `(Ny, Nx+1)`. La face droite est `U[i, j+1]`.
- `V[i, j]` : vitesse sur la face **haute** de la cellule `(i, j)`, taille `(Ny+1, Nx)`. La face basse est `V[i+1, j]`.

### 2a. Le pas de temps (condition CFL)

Avec Euler explicite et upwind, le schéma reste stable si, pour chaque cellule, la quantité de fluide qui sort en un pas de temps ne dépasse pas le contenu de la cellule. Pour une cellule, la somme des vitesses sortantes divisée par la taille de maille doit vérifier `dt * somme <= 1`.

On calcule donc cette somme pour chaque cellule, on prend le maximum, et on pose `dt = cfl / max`, avec `cfl < 1`.

In [ ]:
def pas_de_temps(U, V, dx, dy, cfl):
    # pour chaque cellule : somme des vitesses SORTANTES / taille de maille
    sortant = (np.maximum(U[:, 1:], 0) + np.maximum(-U[:, :-1], 0)) / dx \
            + (np.maximum(V[1:, :], 0) + np.maximum(-V[:-1, :], 0)) / dy
    return cfl / sortant.max()

### 2b. Les flux upwind aux faces

Pour chaque face, il faut la valeur de `c` *à la face* (`c_face`), alors qu'on ne connait que `c` au centre des cellules. L'upwind prend la valeur de la cellule **d'où vient le fluide** :
- face verticale : si `U > 0` le fluide va vers la droite, donc on prend la cellule de gauche, sinon celle de droite,
- face horizontale : si `V > 0` le fluide va vers le bas, donc on prend la cellule du haut, sinon celle du bas.

Le flux est ensuite `F = U * c_face` (et `G = V * c_face`).

**Les bords du tableau** : on entoure `c` d'une ligne et d'une colonne de zéros (`np.pad`), appelées cellules fantômes. Elles servent à avoir une valeur de `c` de l'autre côté des faces du bord. Ici elles valent 0 : s'il y a de l'entrée de fluide par un bord, il apporte c = 0. Dans ce test, V = 0 en haut et en bas donc rien ne passe par là.

In [ ]:
def flux_upwind(c, U, V):
    cp = np.pad(c, 1)                         # cellules fantômes (valeur 0) autour
    # faces verticales : cellule de gauche (cL) et de droite (cR) de chaque face
    cL, cR = cp[1:-1, :-1], cp[1:-1, 1:]
    F = U * np.where(U > 0, cL, cR)
    # faces horizontales : cellule du haut (cT) et du bas (cB) de chaque face
    cT, cB = cp[:-1, 1:-1], cp[1:, 1:-1]
    G = V * np.where(V > 0, cT, cB)
    return F, G

### 2c. Le second membre `dc/dt`

C'est le bilan de flux sur chaque cellule : ce qui entre moins ce qui sort.

`dc/dt = -(F_droite - F_gauche)/dx - (G_bas - G_haut)/dy`

En numpy, `F[:, 1:]` donne les faces droites de toutes les cellules et `F[:, :-1]` les faces gauches. Pareil pour `G`. Chaque flux de face est utilisé **deux fois**, avec un signe opposé, par les deux cellules voisines : c'est ce qui garantit la conservation de la masse.

In [ ]:
def rhs(c, U, V, dx, dy):
    F, G = flux_upwind(c, U, V)
    dcdt = -(F[:, 1:] - F[:, :-1]) / dx - (G[1:, :] - G[:-1, :]) / dy
    return dcdt

## 3. La grille, la tache et les vitesses

- Grille de `Nx = 100` colonnes et `Ny = 50` lignes, mailles de taille 1.
- Tache : gaussienne centrée en `x0 = 25`, `y0 = 25`, d'écart-type `sigma = 5`.
- Vitesse : `U = 1` partout (vers la droite), `V = 0` partout.

On place la tache à gauche et on arrête avant qu'elle atteigne le bord droit, sinon elle sortirait du domaine et la masse diminuerait.

In [ ]:
Nx, Ny = 100, 50
dx = dy = 1.0

# centres des cellules
x = (np.arange(Nx) + 0.5) * dx
y = (np.arange(Ny) + 0.5) * dy
X, Y = np.meshgrid(x, y)                      # tableaux (Ny, Nx)

x0, y0, sigma = 25.0, 25.0, 5.0
def gaussienne(X, Y, x0, y0, sigma):
    return np.exp(-((X - x0)**2 + (Y - y0)**2) / (2 * sigma**2))

c0 = gaussienne(X, Y, x0, y0, sigma)

u0 = 1.0
U = u0 * np.ones((Ny, Nx + 1))
V = np.zeros((Ny + 1, Nx))

## 4. La boucle en temps (Euler explicite)

À chaque pas : on calcule `dc/dt`, puis `c_nouveau = c + dt * dc/dt`. On enregistre la masse totale `somme(c) * dx * dy` à chaque pas pour la vérifier ensuite.

In [ ]:
cfl = 0.5
t_end = 30.0                                   # la tache doit avancer de u0*t_end = 30 cellules
dt0 = pas_de_temps(U, V, dx, dy, cfl)
print("dt =", dt0, " nombre de pas =", int(np.ceil(t_end / dt0)))

c = c0.copy()
t = 0.0
temps, masses = [0.0], [c.sum() * dx * dy]

while t < t_end - 1e-12:
    dt = min(dt0, t_end - t)
    c = c + dt * rhs(c, U, V, dx, dy)
    t += dt
    temps.append(t)
    masses.append(c.sum() * dx * dy)

## 5. Comparaison avec la solution exacte

La solution exacte de `dc/dt + u dc/dx = 0` est la condition initiale translatée de `u * t`. On la calcule et on la compare à notre résultat.

In [ ]:
c_exact = gaussienne(X, Y, x0 + u0 * t_end, y0, sigma)

fig, axs = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, champ, titre in zip(axs, [c0, c, c_exact],
                            ["Initial (t = 0)", f"Calculé (t = {t_end})", f"Exact (t = {t_end})"]):
    im = ax.imshow(champ, vmin=0, vmax=1)
    ax.set_title(titre)
plt.colorbar(im, ax=axs, shrink=0.8)
plt.show()

In [ ]:
# profil horizontal au milieu de la tache (ligne y = y0)
i_mid = Ny // 2
plt.figure(figsize=(8, 4))
plt.plot(x, c0[i_mid, :], '--', label="initial")
plt.plot(x, c[i_mid, :], 'o-', ms=3, label="calculé (upwind)")
plt.plot(x, c_exact[i_mid, :], 'k-', label="exact")
plt.xlabel("x"); plt.ylabel("c"); plt.legend(); plt.grid(True)
plt.show()

**Ce qu'on doit voir** : la tache calculée est au bon endroit (le centre est à `x = 55`), mais elle est **plus basse et plus large** que la solution exacte. C'est la **diffusion numérique** de l'upwind (schéma d'ordre 1) : il étale les fronts, même si l'équation n'a aucune diffusion. Plus la grille est fine, moins c'est visible, c'est ce que mesurera le notebook 4.

## 6. Vérification 1 : la masse est conservée

La masse totale `somme(c) * dx * dy` doit rester constante au cours du temps, presque à la précision machine (variation relative de l'ordre de 1e-12 ou moins). Ici rien ne sort du domaine tant que la tache n'atteint pas le bord droit, donc c'est attendu. Si tu allonges `t_end`, les queues de la gaussienne finiront par sortir et la masse diminuera légèrement : ce n'est pas un bug.

In [ ]:
masses = np.array(masses)
print("masse initiale :", masses[0])
print("masse finale   :", masses[-1])
print("variation relative max :", np.abs(masses - masses[0]).max() / masses[0])

plt.figure(figsize=(8, 3.5))
plt.plot(temps, masses)
plt.xlabel("t"); plt.ylabel("masse totale"); plt.grid(True)
plt.show()

## 7. Vérification 2 : un champ constant reste constant

Si `c = 1` partout et que la vitesse est à divergence nulle, alors `dc/dt` doit être nul partout. C'est un test très efficace pour trouver des erreurs d'indexation dans les flux.

Attention : avec les cellules fantômes à 0, la première colonne reçoit du fluide de c = 0 depuis le bord gauche (`U > 0` sur la face gauche). Le test n'est donc exact que loin du bord entrant. On le fait ici avec des cellules fantômes à 1 pour avoir un vrai test.

In [ ]:
def rhs_const(c, U, V, dx, dy):
    # même calcul que rhs, mais avec des fantômes égaux à 1 (bord entrant = c constant)
    cp = np.pad(c, 1, mode='edge')
    cL, cR = cp[1:-1, :-1], cp[1:-1, 1:]
    F = U * np.where(U > 0, cL, cR)
    cT, cB = cp[:-1, 1:-1], cp[1:, 1:-1]
    G = V * np.where(V > 0, cT, cB)
    return -(F[:, 1:] - F[:, :-1]) / dx - (G[1:, :] - G[:-1, :]) / dy

c_un = np.ones((Ny, Nx))
print("max |dc/dt| pour c = 1 :", np.abs(rhs_const(c_un, U, V, dx, dy)).max())

Le résultat doit être exactement 0 (ou de l'ordre de 1e-16). Sinon, il y a une erreur dans les flux.

## 8. Mesure de l'erreur

On mesure l'écart entre la solution calculée et la solution exacte avec trois normes :
- **L1** : intégrale des écarts absolus,
- **L2** : racine de l'intégrale des carrés,
- **L∞** : écart maximum.

Ces valeurs seront comparées entre maillages dans le notebook 4.

In [ ]:
err = c - c_exact
L1 = np.abs(err).sum() * dx * dy
L2 = np.sqrt((err**2).sum() * dx * dy)
Linf = np.abs(err).max()
print(f"Erreur L1 = {L1:.4e}   L2 = {L2:.4e}   Linf = {Linf:.4e}")
print(f"c max calculé = {c.max():.4f}  (exact : {c_exact.max():.4f})")
print(f"c min calculé = {c.min():.3e}  (l'upwind ne doit pas créer de valeur négative)")

## 9. À toi de jouer

Pour bien comprendre, modifie une chose à la fois et regarde ce qui change :

1. **CFL** : mets `cfl = 0.9`. La tache s'étale moins (en 1D, avec `cfl = 1` l'upwind est exact). Puis `cfl = 1.2` : le calcul explose, c'est l'objet du notebook 3.
2. **Vitesse oblique** : mets `V = 0.5 * np.ones((Ny+1, Nx))`. La tache doit se déplacer en diagonale. Attention au bord du bas : arrête-toi avant qu'elle sorte.
3. **Créneau** : remplace la gaussienne par un carré (`c0 = ((abs(X-x0) < 5) & (abs(Y-y0) < 5)).astype(float)`). Observe comment l'upwind étale les bords raides.
4. **Résolution** : double `Nx` et `Ny` (en adaptant `x0`, `y0`, `sigma` et `t_end` pour garder la même forme physique) et regarde l'erreur diminuer. Ce sera systématisé dans le notebook 4.